# Reproducing: Exploring and Mitigating Adversarial Manipulation of Voting-Based Leaderboards

This notebook demonstrates the core contributions of the paper:
1. De-anonymization of model responses (Section 2)
2. Adversarial voting simulation (Section 3)
3. Mitigations analysis (Section 4)

In [ ]:
import sys
import os
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Set random seed for reproducibility
np.random.seed(42)

## Section 2: De-anonymization of Model Responses

### 2.1 Identity-Probing Detector

In [ ]:
from identity_probing_detector import (
    detect_model_from_response,
    evaluate_identity_probing_detector,
    IDENTITY_PROBING_PROMPTS,
    MODEL_KEYWORDS,
)

# Demonstrate identity-probing detection
test_responses = {
    'claude-3-5-sonnet-20240620': [
        "I'm Claude, an AI assistant made by Anthropic.",
        "I am Claude, created by Anthropic to be helpful.",
        "I'm an AI assistant here to help you.",  # Won't be detected
    ],
    'gpt-4o-2024-05-13': [
        "I'm ChatGPT, an AI assistant made by OpenAI.",
        "I am GPT-4, a large language model by OpenAI.",
        "I'm an AI assistant.",  # Won't be detected
    ],
}

print("Identity-Probing Detection Results:")
print("=" * 50)
for model, responses in test_responses.items():
    accuracy = evaluate_identity_probing_detector(responses, model)
    print(f"{model}: {accuracy * 100:.1f}% accuracy")
    for r in responses:
        detected = detect_model_from_response(r, model)
        print(f"  {'✓' if detected else '✗'} '{r[:60]}...'")

### 2.2 Training-Based Detector

Demonstrates the BoW, TF-IDF, and length-based detectors.

In [ ]:
from training_based_detector import ModelDetector, FEATURE_TYPES

# Create synthetic responses with distinct patterns
# (In practice, these would be real model responses)
np.random.seed(42)

# Simulate Claude-like responses (tend to be structured and use specific vocabulary)
claude_vocab = ['certainly', 'happy', 'assist', 'provide', 'comprehensive', 'overview', 'key', 'aspects']
target_responses = [
    f"I'd be {np.random.choice(claude_vocab)} to help. Here's a {np.random.choice(claude_vocab)} answer: " +
    ' '.join(np.random.choice(claude_vocab, 15))
    for _ in range(50)
]

# Simulate other model responses
other_vocab = ['sure', 'here', 'information', 'note', 'important', 'consider', 'example', 'following']
other_responses = [
    f"Sure! Here's the {np.random.choice(other_vocab)}: " +
    ' '.join(np.random.choice(other_vocab, 15))
    for _ in range(50)
]

# Compare feature types
results = {}
for feature_type in FEATURE_TYPES:
    detector = ModelDetector(feature_type=feature_type, random_state=42)
    result = detector.fit(target_responses, other_responses)
    results[feature_type] = result['test_accuracy']
    print(f"{feature_type}: {result['test_accuracy']:.1f}% accuracy")

# Plot comparison
plt.figure(figsize=(8, 5))
plt.bar(results.keys(), results.values(), color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
plt.axhline(y=50, color='gray', linestyle='--', label='Random baseline')
plt.xlabel('Feature Type')
plt.ylabel('Test Accuracy (%)')
plt.title('Training-Based Detector: Feature Comparison (Table 3)')
plt.legend()
plt.ylim([0, 105])
plt.tight_layout()
plt.show()

## Section 3: Adversarial Voting Simulation

Estimates the number of votes needed to manipulate the leaderboard.

In [ ]:
from adversarial_simulation import (
    create_synthetic_leaderboard,
    AdversarialSimulator,
    SimulationConfig,
)
from bradley_terry import get_rankings

# Create a synthetic leaderboard (approximating Chatbot Arena)
leaderboard = create_synthetic_leaderboard(n_models=20, random_seed=42)

print("Leaderboard (top 5 and bottom 5):")
print("=" * 50)
print("Top 5:")
for rank, name, rating in leaderboard.rankings[:5]:
    print(f"  #{rank}: {name} (rating: {rating:.4f})")
print("Bottom 5:")
for rank, name, rating in leaderboard.rankings[-5:]:
    print(f"  #{rank}: {name} (rating: {rating:.4f})")

In [ ]:
# Simulate attack on the last-ranked model
config = SimulationConfig(
    detection_accuracy=0.95,  # 95% as in paper
    false_positive_rate=0.05,
    false_negative_rate=0.05,
    recalc_interval=1000,
    max_interactions=200000,
    random_seed=42,
)

simulator = AdversarialSimulator(leaderboard, config)

# Try to move the last model up 1, 2, 3 positions
last_model = leaderboard.rankings[-1][1]
last_rank = leaderboard.rankings[-1][0]

print(f"\nAttacking model: {last_model} (rank #{last_rank})")
print("=" * 50)

attack_results = {}
for delta in [1, 2, 3]:
    target_rank = last_rank - delta
    if target_rank < 1:
        continue
    
    # Reset random state
    simulator.rng = np.random.RandomState(42)
    result = simulator.simulate_attack(last_model, target_rank)
    attack_results[delta] = result
    
    if result['achieved']:
        print(f"  Up({delta}): {result['adversarial_votes']} votes, {result['total_interactions']} interactions")
    else:
        print(f"  Up({delta}): Not achieved within {config.max_interactions} interactions")

In [ ]:
# Plot rank history for one attack
if 1 in attack_results and attack_results[1]['achieved']:
    result = attack_results[1]
    rank_history = result['rank_history']
    
    plt.figure(figsize=(10, 5))
    x = np.arange(len(rank_history)) * config.recalc_interval
    plt.plot(x, rank_history, 'b-o', markersize=4)
    plt.axhline(y=last_rank - 1, color='r', linestyle='--', label=f'Target rank #{last_rank - 1}')
    plt.xlabel('Number of Interactions')
    plt.ylabel('Model Rank')
    plt.title(f'Rank History During Attack on {last_model}')
    plt.legend()
    plt.gca().invert_yaxis()  # Lower rank number = better
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Section 4: Mitigations

### 4.1 Attack Cost Model

In [ ]:
from mitigations import compute_attack_cost, estimate_detector_training_cost

# Estimate detector training cost
detector_cost = estimate_detector_training_cost(
    n_prompts=200,
    n_proprietary_models=10,
    n_opensource_models=20,
    responses_per_model=50,
    max_output_tokens=512,
)
print(f"Detector training cost: ${detector_cost:.2f}")
print("(Paper reports ~$440)")

# Cost analysis under different mitigations
print("\nAttack Cost Analysis:")
print("=" * 60)

scenarios = [
    ("No mitigations", float('inf'), 0.0, 0.0),
    ("Rate limit (m=100)", 100, 0.0, 0.0),
    ("Auth + rate limit (m=100, c=$1)", 100, 1.0, 0.0),
    ("Auth + rate limit (m=100, c=$5)", 100, 5.0, 0.0),
    ("CAPTCHA ($0.01/action)", float('inf'), 0.0, 0.01),
    ("CAPTCHA ($0.10/action)", float('inf'), 0.0, 0.10),
]

n_actions = 1000  # ~1000 votes needed
for scenario_name, m, c_account, c_action in scenarios:
    if m == float('inf'):
        import math
        n_accounts = 1
    else:
        import math
        n_accounts = math.ceil(n_actions / m)
    
    cost = compute_attack_cost(n_actions, m if m != float('inf') else n_actions, c_account, c_action, detector_cost)
    print(f"  {scenario_name}: ${cost:.2f} ({n_accounts} accounts)")

### 4.2 Malicious User Detection (Scenario 1)

In [ ]:
from mitigations import MaliciousUserDetector, generate_adversarial_vote_sequence
from bradley_terry import compute_benign_vote_distribution

# Use the synthetic leaderboard
benign_dist = compute_benign_vote_distribution(leaderboard.ratings)
n_models = len(leaderboard.model_names)
target_idx = 0  # Target the top-ranked model

rng = np.random.RandomState(42)

# Generate sequences
n_users = 50
votes_per_user = 50

benign_sequences = [list(rng.choice(n_models, size=votes_per_user, p=benign_dist)) for _ in range(n_users)]
naive_sequences = [generate_adversarial_vote_sequence(target_idx, votes_per_user, n_models, benign_dist, rng=rng) for _ in range(n_users)]
sophisticated_sequences = [generate_adversarial_vote_sequence(target_idx, votes_per_user, n_models, benign_dist, use_public_rankings=True, rng=rng) for _ in range(n_users)]

# Evaluate detector
detector = MaliciousUserDetector(
    benign_vote_distribution=benign_dist,
    model_names=leaderboard.model_names,
    significance_level=0.01,
    n_simulations=1000,
    random_seed=42,
)

naive_results = detector.evaluate_detection(benign_sequences, naive_sequences)
sophisticated_results = detector.evaluate_detection(benign_sequences, sophisticated_sequences)

print("Scenario 1: Known Benign Distribution")
print("=" * 50)
print(f"Naive adversary:")
print(f"  Detection rate (TPR): {naive_results['true_positive_rate']:.1%}")
print(f"  False positive rate: {naive_results['false_positive_rate']:.1%}")
print(f"Sophisticated adversary (uses public rankings):")
print(f"  Detection rate (TPR): {sophisticated_results['true_positive_rate']:.1%}")
print(f"  False positive rate: {sophisticated_results['false_positive_rate']:.1%}")

### 4.3 Malicious User Detection (Scenario 2: Perturbed Leaderboard)

In [ ]:
from mitigations import NeymanPearsonDetector

noise_scales = [0.0, 0.05, 0.1, 0.2, 0.5, 1.0]
detection_rates = []
utility_impacts = []

for noise_scale in noise_scales:
    np_detector = NeymanPearsonDetector(
        true_ratings=leaderboard.ratings,
        model_names=leaderboard.model_names,
        noise_scale=noise_scale,
        random_seed=42,
    )
    
    # Adversary uses perturbed rankings
    adv_sequences = [
        generate_adversarial_vote_sequence(
            target_idx, votes_per_user, n_models,
            np_detector.adversarial_dist, use_public_rankings=True, rng=rng
        )
        for _ in range(n_users)
    ]
    
    results = np_detector.evaluate_detection(benign_sequences, adv_sequences)
    utility = np_detector.compute_utility_impact()
    
    detection_rates.append(results['true_positive_rate'])
    utility_impacts.append(utility)

# Plot (Figures 5 and 6)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(noise_scales, detection_rates, 'b-o', label='Detection rate')
axes[0].set_xlabel('Noise Scale')
axes[0].set_ylabel('Detection Rate (TPR)')
axes[0].set_title('Scenario 2: Detection Rate vs Noise Scale (Figure 5)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1.05])

axes[1].plot(noise_scales, utility_impacts, 'r-s', label='Avg rank change')
axes[1].set_xlabel('Noise Scale')
axes[1].set_ylabel('Average Absolute Rank Change')
axes[1].set_title('Utility Impact of Noise (Figure 6)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nScenario 2 Results:")
print(f"{'Noise Scale':>12} {'Detection Rate':>15} {'Avg Rank Change':>16}")
for ns, dr, ui in zip(noise_scales, detection_rates, utility_impacts):
    print(f"{ns:>12.2f} {dr:>15.1%} {ui:>16.2f}")